<a href="https://colab.research.google.com/github/peakodev/data_science/blob/main/home_05/Hw5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 5

В домашньому завданні до даного модулю ви потренуєтесь робити тестове завдання для влаштування на роботу. За даними акселерометра з мобільного телефону потрібно класифікувати, якою діяльністю займається людина: йде, стоїть, біжить чи йде по сходах. Знайти датасет ви можете за посиланням.

---

Використайте алгоритми SVM та випадковий ліс з бібліотеки scikit-learn. Як характеристики можете брати показники з акселерометра, проте щоб покращити результати роботи алгоритмів, спочатку можна підготувати наш датасет і розрахувати часові ознаки (time domain features). Більше ці характеристики описані в даній статті.

---

Порівняйте результати роботи обох алгоритмів на різних фічах та різні моделі між собою. Використайте метод classification report для порівняння.

---

Порівняння моделей на основі однієї метрики(такої як Accuracy)- не приймається. Дз повинно бути виконано у Jupyter Nootebook,(або Google Colab) і задеплоїне на Гітхаб у вигляді файлу .ipynb.

## Prepare and feature extraction

In [12]:
!unzip /content/homework.zip

Archive:  /content/homework.zip
replace data/idle/idle-1.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [39]:
import os
import pandas as pd
import numpy as np


# Mapping of feature names to their corresponding functions
features = {
    'x_min': lambda df: df['accelerometer_X'].min(),
    'x_max': lambda df: df['accelerometer_X'].max(),
    'x_rms': lambda df: np.sqrt(np.mean(df['accelerometer_X']**2)),
    'x_mean': lambda df: df['accelerometer_X'].mean(),
    'x_std': lambda df: df['accelerometer_X'].std(),
    'x_kurtosis': lambda df: df['accelerometer_X'].kurtosis(),
    'x_median': lambda df: df['accelerometer_X'].median(),
    'y_min': lambda df: df['accelerometer_Y'].min(),
    'y_max': lambda df: df['accelerometer_Y'].max(),
    'y_rms': lambda df: np.sqrt(np.mean(df['accelerometer_Y']**2)),
    'y_mean': lambda df: df['accelerometer_Y'].mean(),
    'y_std': lambda df: df['accelerometer_Y'].std(),
    'y_kurtosis': lambda df: df['accelerometer_Y'].kurtosis(),
    'y_median': lambda df: df['accelerometer_Y'].median(),
    'z_mean': lambda df: df['accelerometer_Z'].mean(),
    'z_kurtosis': lambda df: df['accelerometer_Z'].kurtosis(),
    'z_std': lambda df: df['accelerometer_Z'].std(),
    'z_skewness': lambda df: df['accelerometer_Z'].skew(),
}

# Process data folder and return the result DataFrame
def process_data(data_folder):
    all_rows = []
    for folder_name in os.listdir(data_folder):
        folder_path = os.path.join(data_folder, folder_name)
        if os.path.isdir(folder_path):
            all_rows += process_activity(folder_path, folder_name)

    # Return as a DataFrame
    return pd.DataFrame(all_rows)

# Process each activity (subfolder) and accumulate rows for the DataFrame
def process_activity(folder_path, label):
    rows = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            rows.append(process_file(file_path, label))
    return rows

# Process a single CSV file and extract features
def process_file(file_path, label):
    df = pd.read_csv(file_path)
    row = {feature: features[feature](df) for feature in features.keys()}
    row['label'] = label
    return row

# Run processing on the given folder and store the result in a DataFrame
X = process_data('/content/data')

In [40]:
X.head()

,x_min,x_max,x_rms,x_mean,x_std,x_kurtosis,x_median,y_min,y_max,y_rms,y_mean,y_std,y_kurtosis,y_median,z_mean,z_kurtosis,z_std,z_skewness,label
0,-9.528923,9.035717,3.755184,-0.643721,3.762844,0.867904,-0.569820,-20.159178,-1.489193,11.451708,-10.261708,5.170104,-0.711434,-9.935937,-2.237302,1.015197,5.640395,-0.113161,stairs
1,-6.847417,12.650962,4.063831,1.410025,3.876528,1.815437,1.764527,-16.740260,1.618480,8.492072,-7.309179,4.397240,-0.010538,-7.007828,-1.066377,2.120346,4.554980,0.676529,stairs
2,-9.528923,5.272033,3.805223,-0.410206,3.847720,-0.291449,-0.074220,-25.546135,1.345541,13.529732,-12.038365,6.280627,-0.130276,-11.427525,-2.939122,2.093409,6.334435,-1.231690,stairs
3,-2.011129,12.650962,4.206078,2.495237,3.443868,1.102277,1.982399,-25.828648,1.618480,11.164222,-9.593406,5.807820,1.392355,-8.339004,-1.591665,1.365906,5.405822,1.250945,stairs
4,-9.528923,9.035717,4.198546,-0.452185,4.245483,-0.187162,-0.481234,-25.546135,1.345541,13.699679,-12.159193,6.419413,-0.259483,-11.427525,-2.362279,0.550753,5.946201,0.071427,stairs


## Train with forest

In [44]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X.iloc[:, :-1], X['label'],
                                                    test_size=0.2, random_state=44)

# Створення класифікатора Random Forest
model = RandomForestClassifier()

# Визначення сітки гіперпараметрів для перебору
param_grid = {
    'n_estimators': [100, 200],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [None, 5, 10],
    'min_samples_leaf': [1, 4]
}

# Define multiple scoring metrics
scoring = {
    'accuracy': 'accuracy',
    'precision_macro': 'precision_macro',
    'recall_macro': 'recall_macro'
}

# Створення об'єкта GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid,
                           scoring=scoring, refit='accuracy', return_train_score=True)

# Запуск пошуку найкращих гіперпараметрів
grid_search.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(),
             param_grid={'max_depth': [None, 5, 10],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 4],
                         'n_estimators': [100, 200]},
             refit='accuracy', return_train_score=True,
             scoring={'accuracy': 'accuracy',
                      'precision_macro': 'precision_macro',
                      'recall_macro': 'recall_macro'})

In [46]:
# Вивід найкращих знайдених гіперпараметрів
print("Найкращі параметри:", grid_search.best_params_)

# Accessing scores for different metrics
print("\nGrid search результати по метрикам:")
for metric in scoring:
    print(f"Найкращий {metric}: {grid_search.best_score_:.3f}")

# Оцінка точності моделі з найкращими параметрами
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nТочність моделі на тестовиї даних з найкращими параметрами: {accuracy}")

Найкращі параметри: {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'n_estimators': 200}

Grid search результати по метрикам:
Найкращий accuracy: 0.998
Найкращий precision_macro: 0.998
Найкращий recall_macro: 0.998

Точність моделі на тестовиї даних з найкращими параметрами: 1.0


## SVM

In [52]:
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, precision_score, recall_score

# Define the parameter grid to search over
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf', 'poly', 'sigmoid']
}

scoring_svm = {
    'accuracy': 'accuracy',
    'precision_macro': make_scorer(precision_score, average='macro', zero_division=1),
    'recall_macro': make_scorer(recall_score, average='macro', zero_division=1)
}

# Create a GridSearchCV object
grid = GridSearchCV(SVC(), param_grid, scoring=scoring_svm, refit='accuracy', return_train_score=True)

# Fit the GridSearchCV object to the data
grid.fit(X_train, y_train)

GridSearchCV(estimator=SVC(),
             param_grid={'C': [0.1, 1, 10, 100], 'gamma': [1, 0.1, 0.01, 0.001],
                         'kernel': ['rbf', 'poly', 'sigmoid']},
             refit='accuracy', return_train_score=True,
             scoring={'accuracy': 'accuracy',
                      'precision_macro': make_scorer(precision_score, response_method='predict', average=macro, zero_division=1),
                      'recall_macro': make_scorer(recall_score, response_method='predict', average=macro, zero_division=1)})

In [53]:
# Вивід найкращих знайдених гіперпараметрів
print("Найкращі параметри:", grid.best_params_)

# Accessing scores for different metrics
print("\nGrid search результати по метрикам:")
for metric in scoring:
    print(f"Найкращий {metric}: {grid.best_score_:.3f}")

# Оцінка точності моделі з найкращими параметрами
best_model_SVM = grid.best_estimator_
y_pred = best_model_SVM.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nТочність моделі на тестовиї даних з найкращими параметрами: {accuracy}")

Найкращі параметри: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}

Grid search результати по метрикам:
Найкращий accuracy: 0.999
Найкращий precision_macro: 0.999
Найкращий recall_macro: 0.999

Точність моделі на тестовиї даних з найкращими параметрами: 1.0


## Clasification Report

In [54]:
from sklearn.metrics import classification_report

models = {
    "RandomForest": best_model,
    "SVM": best_model_SVM,
}

for model_name, model in models.items():
    y_pred = model.predict(X_test)

    print(f"Classification Report for {model_name}:\n")
    print(classification_report(y_test, y_pred, target_names=['iddle', 'running', 'stairs', 'walking']))
    print("-" * 60)

Classification Report for RandomForest:

              precision    recall  f1-score   support

       iddle       1.00      1.00      1.00       197
     running       1.00      1.00      1.00       696
      stairs       1.00      0.96      0.98        26
     walking       1.00      1.00      1.00       374

    accuracy                           1.00      1293
   macro avg       1.00      0.99      0.99      1293
weighted avg       1.00      1.00      1.00      1293

------------------------------------------------------------
Classification Report for SVM:

              precision    recall  f1-score   support

       iddle       1.00      1.00      1.00       197
     running       1.00      1.00      1.00       696
      stairs       1.00      1.00      1.00        26
     walking       1.00      1.00      1.00       374

    accuracy                           1.00      1293
   macro avg       1.00      1.00      1.00      1293
weighted avg       1.00      1.00      1.00      12

Висновок: SVM {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'} найкраща модель